In [71]:
!pip install datasets pandas matplotlib seaborn spacy
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 330.3 kB/s eta 0:00:39
     --------------------------------------- 0.0/12.8 MB 330.3 kB/s eta 0:00:39
     --------------------------------------- 0.0/12.8 MB 330.3 kB/s eta 0:00:39
     --------------------------------------- 0.1/12.8 MB 297.7 kB/s eta 0:00:43
     --------------------------------------- 0.1/12.8 MB 245.8 kB/s eta 0:00:52
     --------------------------------------- 0.1/12.8 MB 245.8 kB/s eta 0:00:52
     --------------------------------------- 0.1/12.8 MB 245.8 kB/s eta 0:00:52
     --------------------------------------- 0.1/12.8 MB 262.6 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 262.6 kB/s eta 0:00:49
     --------------------------------------- 0.1/12.8 MB 283.8 k

In [73]:
from datasets import load_dataset
import pandas as pd
import re
import spacy
import json

In [75]:
path = "C:\\Users\\HP\\Desktop\\KMIT\\Sem\\Entity pulse\\FinEntity.json"
df = pd.read_json(path)
print(df.head())
print(df.columns)

                                             content  \
0  Johnson & Johnson <JNJ.N> shares gained 0.20% ...   
1  On the positive side, Siemens is rallying 6% a...   
2  Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
3  Nearly all major S&P 500 sectors are red, with...   
4  NEW YORK - Wall Street ended sharply higher on...   

                                         annotations  
0  [{'end': 17, 'tag': 'Positive', 'value': 'John...  
1  [{'end': 107, 'tag': 'Positive', 'value': 'Huh...  
2  [{'end': 11, 'tag': 'Positive', 'value': 'Bren...  
3  [{'end': 162, 'tag': 'Positive', 'value': 'hea...  
4  [{'end': 69, 'tag': 'Positive', 'value': 'Tesl...  
Index(['content', 'annotations'], dtype='object')


In [95]:
data = []
for _, row in df.iterrows():
    sentence = row['content']
    for ann in row['annotations']:
        entity = ann.get('value')
        sentiment = ann.get('tag')
        data.append([sentence, entity, sentiment])

df_flat = pd.DataFrame(data, columns=['sentence', 'entity', 'sentiment'])
print("\nFlattened dataset sample:\n", df_flat.head(20))


Flattened dataset sample:
                                              sentence  \
0   Johnson & Johnson <JNJ.N> shares gained 0.20% ...   
1   On the positive side, Siemens is rallying 6% a...   
2   On the positive side, Siemens is rallying 6% a...   
3   Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
4   Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
5   Nearly all major S&P 500 sectors are red, with...   
6   Nearly all major S&P 500 sectors are red, with...   
7   Nearly all major S&P 500 sectors are red, with...   
8   Nearly all major S&P 500 sectors are red, with...   
9   Nearly all major S&P 500 sectors are red, with...   
10  NEW YORK - Wall Street ended sharply higher on...   
11  NEW YORK - Wall Street ended sharply higher on...   
12  Levi Strauss & Co <LEVI.N> gained 2% after its...   
13  However, top oilfield services company Schlumb...   
14      JPMorgan Chase & Co <JPM.N>, Morgan Stanle...   
15      JPMorgan Chase & Co <JPM.N>, Morgan Stanle...   
16 

In [79]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[A-Z]+\.[A-Z]+>', '', text)
    text = re.sub(r'[^A-Za-z0-9.,!?;\'\"\s]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nBefore cleaning:\n", df_flat['sentence'].head(5))
df_flat['sentence'] = df_flat['sentence'].apply(clean_text)
print("\nAfter cleaning:\n", df_flat['sentence'].head(5))


Before cleaning:
 0    Johnson & Johnson <JNJ.N> shares gained 0.20% ...
1    On the positive side, Siemens is rallying 6% a...
2    On the positive side, Siemens is rallying 6% a...
3    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
4    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
Name: sentence, dtype: object

After cleaning:
 0    Johnson Johnson shares gained 0.20 after posti...
1    On the positive side, Siemens is rallying 6 af...
2    On the positive side, Siemens is rallying 6 af...
3    Brent crude rose 1.4 to 100.69 per barrel and ...
4    Brent crude rose 1.4 to 100.69 per barrel and ...
Name: sentence, dtype: object


In [81]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

df_flat['sentence'] = df_flat['sentence'].str.lower().apply(lemmatize_text)

In [83]:
before_drop = len(df_flat)
df_flat.dropna(subset=['sentence'], inplace=True)
df_flat = df_flat[df_flat['sentence'].str.strip() != ""]
after_drop = len(df_flat)
print(f"\nDropped {before_drop - after_drop} empty rows. Remaining rows: {after_drop}")


Dropped 0 empty rows. Remaining rows: 2131


In [85]:
ENTITY_TYPES = [
    "COMPANY",
    "ORGANIZATION",
    "CURRENCY",
    "PERSON",
    "OTHER"
]

In [153]:
def get_entity_type(entity):
    entity_lower = entity.lower()

    if "inc" in entity_lower or "ltd" in entity_lower or "corp" in entity_lower:
        return "COMPANY"
    elif any(keyword in entity_lower for keyword in ["organization", "association", "committee", "foundation", "agency"]):
        return "ORGANIZATION"
    elif entity_lower in ["usd", "eur", "gbp", "inr"]:
        return "CURRENCY"
    elif all(word.isalpha() and word.istitle() for word in entity.split()):
        return "PERSON"
    else:
        return "OTHER"

df_flat['entity_type'] = df_flat['entity'].apply(lambda x: get_entity_type(str(x)))

In [157]:
def get_entity_type_info(entity):
    entity_type = get_entity_type(entity)                       
    return entity_type

entity = "World Health Organization"
etype = get_entity_type_info(entity)
print(f"Entity: {entity}")
print(f"Entity Type: {etype}")

Entity: World Health Organization
Entity Type: ORGANIZATION


In [97]:
SENTIMENTS = ["positive", "negative", "neutral"]
TAGS = ["B","I","L","O","U"]
all_labels = []
for ent_type in ENTITY_TYPES:
    for sent in SENTIMENTS:
        all_labels.append(f"{ent_type}-{sent}")

label2id = {label: idx for idx, label in enumerate(all_labels)}
id2label = {idx: label for label, idx in label2id.items()}
entity_type2id = {etype: idx for idx, etype in enumerate(ENTITY_TYPES)}
id2entity_type = {idx: etype for etype, idx in entity_type2id.items()}
mappings = {
    "label2id": label2id,                     # entity-sentiment combined label
    "id2label": id2label,
    "entity_type2id": entity_type2id,         # only entity types
    "id2entity_type": id2entity_type
}

print("\nSample label2id mapping (first 10):", dict(list(label2id.items())[:10]))
print(f"Total Labels: {len(all_labels)}")


Sample label2id mapping (first 10): {'COMPANY-positive': 0, 'COMPANY-negative': 1, 'COMPANY-neutral': 2, 'ORGANIZATION-positive': 3, 'ORGANIZATION-negative': 4, 'ORGANIZATION-neutral': 5, 'CURRENCY-positive': 6, 'CURRENCY-negative': 7, 'CURRENCY-neutral': 8, 'PERSON-positive': 9}
Total Labels: 15


In [103]:
output_path = "Preprocessed_FinEntity.csv"
df_flat.to_csv(output_path, index=False)
print(f"\nPreprocessing complete! Saved to {output_path}")

with open("label_mappings.json", "w") as f:
    json.dump(mappings, f, indent=4)
print("\nLabel schema & mappings saved to label_mappings.json")


Preprocessing complete! Saved to Preprocessed_FinEntity.csv

Label schema & mappings saved to label_mappings.json
